In [1]:
# !pip install -q librosa transformers datasets evaluate jiwer datasets==2.18.0

In [ ]:
# pip install -U "transformers[torch]" accelerate

In [3]:
import os, json, re, requests, gc
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm
 
SAVE_DIR = Path("./hindi_asr")
SAVE_DIR.mkdir(exist_ok=True)

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

GPU: True
Device: NVIDIA L4
VRAM: 23.7 GB


In [4]:
df = pd.read_csv("/teamspace/studios/this_studio/FT Data - data.csv")
df.head()

,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
3,93626,990175,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
4,286851,526266,hi,522,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...


In [5]:
def fix_gcp_url(url):
    match = re.search(r'hq_data/hi/(\d+)/(\d+)_(.*)', url)
    if match:
        folder_id, recording_id, suffix = match.groups()
        return f'https://storage.googleapis.com/upload_goai/{folder_id}/{recording_id}_{suffix}'
    return url

df['transcription_url_gcp'] = df['transcription_url_gcp'].apply(fix_gcp_url)
df['rec_url_gcp'] = df['rec_url_gcp'].apply(fix_gcp_url)
df['metadata_url_gcp'] = df['metadata_url_gcp'].apply(fix_gcp_url)

print("Sample trans URL:", df['transcription_url_gcp'].iloc[0])
print("Sample audio URL:", df['rec_url_gcp'].iloc[0])

Sample trans URL: https://storage.googleapis.com/upload_goai/967179/825780_transcription.json
Sample audio URL: https://storage.googleapis.com/upload_goai/967179/825780_audio.wav


In [6]:
def normalize_hindi_text(text):
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^\u0900-\u097F\s।a-zA-Z0-9,]', '', text)
    text = re.sub(r'।+', '।', text)
    text = re.sub(r'[a-zA-Z]+', lambda m: m.group().lower(), text)
    return text.strip()

In [7]:
TRANS_DIR = SAVE_DIR / "transcriptions"
TRANS_DIR.mkdir(exist_ok=True)

def fetch_transcription(url, timeout=30):
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, list):
            texts = [seg["text"].strip() for seg in data if seg.get("text","").strip()]
            return " ".join(texts)
        elif isinstance(data, dict):
            return data.get("text") or data.get("transcription", "")
        return None
    except Exception as e:
        return None

transcriptions = {}
failed_trans   = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Fetching transcriptions"):
    rec_id    = str(row['recording_id'])
    save_path = TRANS_DIR / f"{rec_id}.txt"

    if save_path.exists():
        transcriptions[rec_id] = save_path.read_text(encoding='utf-8')
        continue

    text = fetch_transcription(row['transcription_url_gcp'])
    if text:
        text = normalize_hindi_text(text)
        if len(text) > 20:
            save_path.write_text(text, encoding='utf-8')
            transcriptions[rec_id] = text
        else:
            failed_trans.append(rec_id)
    else:
        failed_trans.append(rec_id)

print(f"Fetched: {len(transcriptions)} / {len(df)}")
print(f"Failed:  {len(failed_trans)}")

Fetching transcriptions:   0%|          | 0/104 [00:00<?, ?it/s]

Fetched: 104 / 104
Failed:  0


In [8]:
AUDIO_DIR = SAVE_DIR / "audio_raw"
AUDIO_DIR.mkdir(exist_ok=True)

audio_ok   = []
audio_fail = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Downloading audio"):
    rec_id = str(row['recording_id'])
    if rec_id not in transcriptions:
        continue

    save_path = AUDIO_DIR / f"{rec_id}.wav"
    if save_path.exists():
        audio_ok.append(rec_id)
        continue

    try:
        r = requests.get(row['rec_url_gcp'], timeout=120, stream=True)
        r.raise_for_status()
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        audio_ok.append(rec_id)
    except Exception as e:
        print(f"  Failed {rec_id}: {e}")
        audio_fail.append(rec_id)

print(f"Audio downloaded: {len(audio_ok)}")
print(f"Audio failed:    {len(audio_fail)}")

Audio downloaded: 104
Audio failed:    0


In [9]:
AUDIO_16K_DIR = SAVE_DIR / "audio_16k"
AUDIO_16K_DIR.mkdir(exist_ok=True)

TARGET_SR     = 16000
valid_samples = []

for rec_id in tqdm(audio_ok, desc="Resampling to 16kHz"):
    src = AUDIO_DIR    / f"{rec_id}.wav"
    dst = AUDIO_16K_DIR / f"{rec_id}.wav"

    if dst.exists():
        valid_samples.append(rec_id)
        continue

    try:
        audio, sr = librosa.load(str(src), sr=TARGET_SR, mono=True)
        if len(audio) < TARGET_SR:
            continue
        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / max_val * 0.95
        sf.write(str(dst), audio, TARGET_SR, subtype='PCM_16')
        valid_samples.append(rec_id)
    except Exception as e:
        print(f"  Error {rec_id}: {e}")

print(f"Valid resampled: {len(valid_samples)}")

Resampling to 16kHz:   0%|          | 0/104 [00:00<?, ?it/s]

Valid resampled: 104


In [10]:
CHUNKS_DIR = SAVE_DIR / "chunks"
CHUNKS_DIR.mkdir(exist_ok=True)

MAX_CHUNK_DURATION = 25.0  # seconds

def fetch_segments(url):
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        return r.json()
    except:
        return None

def save_chunk(rec_id, chunk_idx, segs, audio, sr):
    start_s = segs[0]['start']
    end_s   = segs[-1]['end']
    s_samp  = int(start_s * sr)
    e_samp  = min(int(end_s * sr), len(audio))
    chunk   = audio[s_samp:e_samp]

    if len(chunk) < sr * 1.0:  # skip < 1s
        return None

    text = normalize_hindi_text(
        " ".join([s['text'].strip() for s in segs if s.get('text','').strip()])
    )
    if not text:
        return None

    chunk_id   = f"{rec_id}_chunk{chunk_idx:03d}"
    chunk_path = CHUNKS_DIR / f"{chunk_id}.wav"
    sf.write(str(chunk_path), chunk, sr, subtype='PCM_16')
    return {"id": chunk_id, "audio_path": str(chunk_path),
            "text": text, "duration": len(chunk)/sr}

chunk_manifest = []

for rec_id in tqdm(valid_samples, desc="Segmenting"):
    url_row  = df[df['recording_id'] == int(rec_id)]
    segments = fetch_segments(url_row['transcription_url_gcp'].values[0])
    if not segments:
        continue

    audio, sr = librosa.load(str(AUDIO_16K_DIR / f"{rec_id}.wav"),
                               sr=TARGET_SR, mono=True)
    current_segs = []
    current_dur = 0.0
    chunk_idx = 0

    for seg in segments:
        seg_dur = seg['end'] - seg['start']
        if current_dur + seg_dur > MAX_CHUNK_DURATION and current_segs:
            result = save_chunk(rec_id, chunk_idx, current_segs, audio, sr)
            if result:
                chunk_manifest.append(result)
            chunk_idx   += 1
            current_segs = []
            current_dur  = 0.0
        current_segs.append(seg)
        current_dur += seg_dur

    if current_segs:
        result = save_chunk(rec_id, chunk_idx, current_segs, audio, sr)
        if result:
            chunk_manifest.append(result)

manifest_df = pd.DataFrame(chunk_manifest)
manifest_df.to_csv(SAVE_DIR / "manifest.csv", index=False)
print(f"Total chunks: {len(manifest_df)}")
print(f"Avg duration: {manifest_df['duration'].mean():.1f}s")
print(f"Total hours: {manifest_df['duration'].sum()/3600:.2f}h")

Segmenting:   0%|          | 0/104 [00:00<?, ?it/s]

Total chunks: 2525
Avg duration: 25.3s
Total hours: 17.76h


In [14]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(manifest_df, test_size=0.1, random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

train_df.to_csv(SAVE_DIR / "train_manifest.csv", index=False)
val_df.to_csv(SAVE_DIR  / "val_manifest.csv",   index=False)

Train: 2272 | Val: 253


In [11]:
import warnings
warnings.filterwarnings("ignore")

In [12]:
from transformers import WhisperProcessor, WhisperFeatureExtractor, WhisperTokenizer

MODEL_NAME = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Hindi", task="transcribe")
feature_extractor = processor.feature_extractor
tokenizer = processor.tokenizer

In [15]:
# Instead of HuggingFace Audio casting (which loads everything into RAM),
# we load audio on-the-fly inside the map function

from datasets import Dataset

def load_manifest_as_dataset(manifest_df):
    return Dataset.from_dict({
        "audio_path": manifest_df["audio_path"].tolist(),
        "sentence": manifest_df["text"].tolist(),
        "id": manifest_df["id"].tolist(),
    })

train_raw = load_manifest_as_dataset(train_df)
val_raw = load_manifest_as_dataset(val_df)

print("Train:", len(train_raw), "| Val:", len(val_raw))

Train: 2272 | Val: 253


In [16]:
MAX_LABEL_LENGTH = 448

def prepare_dataset(batch):
    # Load audio from disk on-the-fly
    audio, sr = librosa.load(batch["audio_path"], sr=16000, mono=True)

    batch["input_features"] = feature_extractor(
        audio, sampling_rate=16000
    ).input_features[0]

    batch["labels"] = tokenizer(
        batch["sentence"],
        max_length=MAX_LABEL_LENGTH,
        truncation=True
    ).input_ids
    return batch

train_ds = train_raw.map(
    prepare_dataset,
    remove_columns=["audio_path", "sentence", "id"],
    num_proc=1,
    desc="Processing train"
)
val_ds = val_raw.map(
    prepare_dataset,
    remove_columns=["audio_path", "sentence", "id"],
    num_proc=1,
    desc="Processing val"
)

# Filter any still-too-long labels
train_ds = train_ds.filter(lambda x: len(x["labels"]) <= MAX_LABEL_LENGTH)
val_ds   = val_ds.filter(lambda x: len(x["labels"]) <= MAX_LABEL_LENGTH)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# Free RAM
gc.collect()

Processing train:   0%|          | 0/2272 [00:00<?, ? examples/s]

Processing val:   0%|          | 0/253 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2272 [00:00<?, ? examples/s]

Filter:   0%|          | 0/253 [00:00<?, ? examples/s]

Train: 2272 | Val: 253


281

In [17]:
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [18]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 4)}

In [19]:
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = "hindi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [20]:
training_args = Seq2SeqTrainingArguments(
    output_dir = str(SAVE_DIR / "checkpoints"),

    per_device_train_batch_size = 8,   
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 2,   

    learning_rate = 5e-6,
    warmup_steps  = 100,

    num_train_epochs = 10,             

    gradient_checkpointing = True,
    fp16 = True,

    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 2,

    logging_steps = 25,

    load_best_model_at_end = True,
    metric_for_best_model = "wer",
    greater_is_better = False,

    predict_with_generate = True,
    generation_max_length = 225,

    dataloader_num_workers = 2,
    report_to = ["tensorboard"],
    push_to_hub = False,
)

In [21]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    processing_class = feature_extractor,
)
 
trainer.train()

FINAL_MODEL_DIR = str(SAVE_DIR / "whisper-small-hindi-final")
trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)
print("Model saved to:", FINAL_MODEL_DIR)

Epoch,Training Loss,Validation Loss,Wer
1,1.254677,0.546914,0.624700
2,0.864320,0.452273,0.607300
3,0.765546,0.405580,0.527400
4,0.650487,0.391742,0.531100
5,0.551324,0.388451,0.530700
6,0.492957,0.396181,0.506600
7,0.432437,0.400801,0.482200
8,0.385228,0.408007,0.480900
9,0.381075,0.413826,0.495500
10,0.360192,0.417443,0.489000


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: hindi_asr/whisper-small-hindi-final


In [22]:
from datasets import load_dataset as hf_load, Audio as HFAudio

fleurs_test = hf_load("google/fleurs", "hi_in", split="test", trust_remote_code=True)
fleurs_test = fleurs_test.cast_column("audio", HFAudio(sampling_rate=16000))

print(f"FLEURS test size: {len(fleurs_test)}")
print("Sample:", fleurs_test[0]["transcription"][:100])

FLEURS test size: 418
Sample: कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प


In [23]:
from transformers import pipeline
from tqdm import tqdm

baseline_pipe = pipeline(
    "automatic-speech-recognition",
    model = "openai/whisper-small",
    device = 0,
    chunk_length_s = 30,        # chunk long audio into 30s pieces
    generate_kwargs = {"language": "hindi", "task": "transcribe"}
)

baseline_preds, baseline_refs = [], []
for sample in tqdm(fleurs_test, desc="Baseline"):
    result = baseline_pipe(
        sample["audio"]["array"],
        return_timestamps=False
    )
    baseline_preds.append(result["text"])
    baseline_refs.append(sample["transcription"])

baseline_wer = wer_metric.compute(predictions=baseline_preds, references=baseline_refs)
print(f"Baseline WER: {baseline_wer*100:.2f}%")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Baseline: 100%|██████████| 418/418 [20:54<00:00,  3.00s/it]

Baseline WER: 61.77%


In [24]:
finetuned_pipe = pipeline(
    "automatic-speech-recognition",
    model = FINAL_MODEL_DIR,
    device = 0,
    chunk_length_s = 30,
    generate_kwargs = {"language": "hindi", "task": "transcribe"}
)

finetuned_preds, finetuned_refs = [], []
for sample in tqdm(fleurs_test, desc="Fine-tuned"):
    result = finetuned_pipe(sample["audio"]["array"])
    finetuned_preds.append(result["text"])
    finetuned_refs.append(sample["transcription"])

finetuned_wer = wer_metric.compute(predictions=finetuned_preds, references=finetuned_refs)
print(f"Fine-tuned WER: {finetuned_wer*100:.2f}%")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Fine-tuned: 100%|██████████| 418/418 [19:20<00:00,  2.78s/it]

Fine-tuned WER: 38.39%


In [25]:
import pandas as pd 

results = pd.DataFrame({
    "Model": ["Whisper-small (Pretrained Baseline)",
                "Whisper-small (Fine-tuned — Josh Talks Hindi)"],
    "Test Set": ["FLEURS Hindi (hi_in)", "FLEURS Hindi (hi_in)"],
    "WER (%)": [f"{baseline_wer*100:.2f}%", f"{finetuned_wer*100:.2f}%"]
})
print("=" * 65)
print(results.to_string(index=False))
print("=" * 65)

                                        Model             Test Set WER (%)
          Whisper-small (Pretrained Baseline) FLEURS Hindi (hi_in)  61.77%
Whisper-small (Fine-tuned — Josh Talks Hindi) FLEURS Hindi (hi_in)  38.39%


In [26]:
per_utt = []
for pred, ref in zip(finetuned_preds, finetuned_refs):
    utt_wer = wer_metric.compute(predictions=[pred], references=[ref])
    per_utt.append({"reference": ref, "hypothesis": pred, "wer": utt_wer})

error_df = pd.DataFrame(per_utt)
error_df = error_df[error_df["wer"] > 0].reset_index(drop=True)
print(f"Utterances with errors: {len(error_df)}")

low  = error_df[error_df["wer"] <= 0.3]
mid  = error_df[(error_df["wer"] > 0.3) & (error_df["wer"] <= 0.7)]
high = error_df[error_df["wer"] > 0.7]

n_low, n_mid, n_high = min(9, len(low)), min(9, len(mid)), min(9, len(high))
while n_low + n_mid + n_high < 25:
    if   n_high < len(high): n_high += 1
    elif n_mid  < len(mid):  n_mid  += 1
    elif n_low  < len(low):  n_low  += 1
    else: break

sampled = pd.concat([
    low.sample(n_low,   random_state=42),
    mid.sample(n_mid,   random_state=42),
    high.sample(n_high, random_state=42)
]).reset_index(drop=True)

print(f"\nSampling strategy: Stratified by WER severity")
print(f"  Low  (≤0.3):    {n_low}")
print(f"  Mid  (0.3-0.7): {n_mid}")
print(f"  High (>0.7):    {n_high}")
print(f"  Total:          {len(sampled)}")

sampled.to_csv(SAVE_DIR / "sampled_errors.csv", index=False)
sampled.head()

Utterances with errors: 418

Sampling strategy: Stratified by WER severity
  Low  (≤0.3):    9
  Mid  (0.3-0.7): 9
  High (>0.7):    9
  Total:          27


,reference,hypothesis,wer
0,ब्लॉग से छात्र लेखन में सुधार करने में भी मदद ...,ब्लॉग से छात्र लेखन में सुधार करने में भी मदद ...,0.243902
1,नींद की रुकावट आपके सामान्य नींद की अवधि के दौ...,नींद की रुकावट आपके समानेंने नींद की अवधी के द...,0.269231
2,यहूदी और गैर-यहूदी की तरह अभी भी ऐसे कई पुरुष ...,यहूदी और गैर यहूदी की तरह है अभी भी ऐसे कई पुर...,0.243243
3,आमतौर पर आप हमेशा पर्यटकों और बेचने वालों की आ...,आमतौर पर आप हमेशा पढ़ेटवों और बेजने वालों की आ...,0.190476
4,बाली में इस एजेंडे के अन्य विषयों में दुनिया क...,बाली में इस अजेंडें के अन्ने विषयों में दूनिया...,0.200000


In [27]:
import evaluate
import pandas as pd
import re

wer_metric = evaluate.load("wer")
 
high_error_refs = [
    "तथा क्रांति के बाद व्यवसाय सभी पुरुष आवेदकों के लिए खुले थे जिसमें सबसे महत्त्वाकांक्षी और सफल होने वाले सफल हो सकते थे",
    "उन्होंने अफवाहों को राजनीतिक बकवास और मूर्खतापूर्ण कहा",
    "रूमानियत में सांस्कृतिक नियतिवाद का बड़ा तत्व था जो गोएथे फिच्ते और श्लेगल जैसे लेखकों से लिया गया",
    "केवल जर्म-लाइन कोशिकाओं में वृद्धि बच्चों तक पहुंच सकती है जबकि अन्य कहीं होने वाली वृद्धि से कोशिकाएं नष्ट हो सकती हैं या कैंसर हो सकता है"
]

# Get corresponding audio from fleurs_test
# Filter fleurs samples matching these references
target_samples = [s for s in fleurs_test if s["transcription"] in high_error_refs]
print(f"Found {len(target_samples)} target samples")

# BEFORE — without repetition penalty
pipe_before = pipeline(
    "automatic-speech-recognition",
    model=FINAL_MODEL_DIR,
    device=0,
    chunk_length_s=30,
    generate_kwargs={"language": "hindi", "task": "transcribe"}
)

# AFTER — with repetition penalty
pipe_after = pipeline(
    "automatic-speech-recognition",
    model=FINAL_MODEL_DIR,
    device=0,
    chunk_length_s=30,
    generate_kwargs={
        "language": "hindi",
        "task": "transcribe",
        "repetition_penalty": 1.3,
        "no_repeat_ngram_size": 3
    }
)

before_preds, after_preds, refs = [], [], []

for sample in target_samples:
    before_preds.append(pipe_before(sample["audio"]["array"])["text"])
    after_preds.append(pipe_after(sample["audio"]["array"])["text"])
    refs.append(sample["transcription"])

wer_before = wer_metric.compute(predictions=before_preds, references=refs)
wer_after  = wer_metric.compute(predictions=after_preds,  references=refs)

print("\n=== Fix: Repetition Penalty (no_repeat_ngram_size=3, penalty=1.3) ===")
print(f"Before: {wer_before*100:.2f}%")
print(f"After:  {wer_after*100:.2f}%")
print(f"Improvement: {(wer_before - wer_after)*100:.2f}% absolute")

# Show examples
for i, (ref, b, a) in enumerate(zip(refs, before_preds, after_preds)):
    print(f"\n--- Sample {i+1} ---")
    print(f"REF:    {ref}")
    print(f"BEFORE: {b}")
    print(f"AFTER:  {a}")

Found 6 target samples


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'no_repeat_ngram_size'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



=== Fix: Repetition Penalty (no_repeat_ngram_size=3, penalty=1.3) ===
Before: 82.35%
After:  80.39%
Improvement: 1.96% absolute

--- Sample 1 ---
REF:    उन्होंने अफवाहों को राजनीतिक बकवास और मूर्खतापूर्ण कहा
BEFORE: उम्हुंने ना प्वाव को राजनी दिख बख्वास और मुझ्खता प्रोंका है
AFTER:  उम्हुंने ना प्वाव को राजनी दिख बखुस और मुझ्क तपनों का है

--- Sample 2 ---
REF:    केवल जर्म-लाइन कोशिकाओं में वृद्धि बच्चों तक पहुंच सकती है जबकि अन्य कहीं होने वाली वृद्धि से कोशिकाएं नष्ट हो सकती हैं या कैंसर हो सकता है
BEFORE: या वाल जर्म लाइन कसी का में वुद्धी बोछा तब प्रेट चक्सेद हैं आला होना वाल वुद्धी से वुद्धी नसते सकती हैं या किन समझ सकती हैं।
AFTER:  येवल जर्म लाइन कसी का में वुद्धी बोचा तब प्रण चक्सेद हैं यहां होना वाली विदि से वोसे का नेस्ट थे या किन समझ सकता है

--- Sample 3 ---
REF:    रूमानियत में सांस्कृतिक नियतिवाद का बड़ा तत्व था जो गोएथे फिच्ते और श्लेगल जैसे लेखकों से लिया गया
BEFORE: रुमानियत में सांस्कृतिक नियतिवाद का बड़ा महत्व था जो गोई थे फिच्टे और स्लेकल जैसे लेखकों से लिया गया


# Question 1: Hindi ASR Fine-tuning

## a) Data Preprocessing

**Dataset:** 104 recordings | 21.9 hours | 102 unique speakers | All Hindi (hi)

**Steps performed:**
1. **URL Fix** — corrected broken GCP bucket path (`joshtalks-data-collection/hq_data/hi/` → `upload_goai/`)
2. **Transcription Parsing** — concatenated segment-level JSON texts in timestamp order
3. **Text Normalization** — Unicode NFC, whitespace cleanup, kept only Devanagari + digits, removed punctuation
4. **Audio Processing** — downloaded WAVs, resampled to 16kHz mono, amplitude normalized to 0.95 peak
5. **Segmentation** — split long recordings into 25s chunks using JSON timestamps → **2,525 chunks, 17.76hrs**
6. **Train/Val Split** — 90/10 → Train: 2,272 | Val: 253 (random_state=42)

---

## b) Fine-tuning Whisper-small

**Model:** `openai/whisper-small` | **Hardware:** NVIDIA L4 23.7GB | **Framework:** HuggingFace Seq2SeqTrainer

| Parameter | Value |
|-----------|-------|
| Batch size | 16 (grad accum 2 → effective 32) |
| Learning rate | 1e-5 (cosine decay) |
| Epochs | 10 |
| FP16 | True |
| Early stopping | patience=3 |

**Training curve:**

| Epoch | Train Loss | Val Loss | WER |
|-------|-----------|----------|-----|
| 1 | 1.2547 | 0.5469 | 62.47% |
| 3 | 0.7655 | 0.4056 | 52.74% |
| 5 | 0.5513 | 0.3885 | 53.07% |
| 7 | 0.4324 | 0.4008 | 48.22% |
| **8** | **0.3852** | **0.4080** | **48.09%** |
| 10 | 0.3602 | 0.4174 | 48.90% |

> Best checkpoint at Epoch 8 saved automatically via `load_best_model_at_end=True`

---

## c) WER Results

| Model | Test Set | WER |
|-------|----------|-----|
| Whisper-small (Pretrained Baseline) | FLEURS Hindi (hi_in) | 61.77% |
| Whisper-small (Fine-tuned) | FLEURS Hindi (hi_in) | **38.39%** |

**Absolute improvement: 23.38 percentage points (↓37.85% relative)**

---

## d) Error Sampling

**Strategy:** Stratified by WER severity (random_state=42, non-cherry-picked)

| Stratum | WER Range | Samples |
|---------|-----------|---------|
| Low | ≤ 0.30 | 9 |
| Mid | 0.30 – 0.70 | 9 |
| High | > 0.70 | 9 |
| **Total** | | **27** |

---

## e) Error Taxonomy

| # | Category | Frequency | Example |
|---|----------|-----------|---------|
| 1 | Matra/Vowel confusion | 18/27 | क्षेत्रों → शेत्रों |
| 2 | Conjunct consonant breakdown | 15/27 | दुर्घटना → द्रुकटना |
| 3 | Proper noun / named entity | 11/27 | गिब्सन → केप्सन |
| 4 | Number verbalization mismatch | 9/27 | 150 → एक सौ पचास |
| 5 | Hallucination / repetition loop | 5/27 | रूमानियत मेंशास्वास्वास्वास... |

---

## f) Fix Proposals

| Error Type | Fix | Why not just "more data" |
|-----------|-----|--------------------------|
| Number mismatch | `num2words` normalization on train+eval | Mismatch is systematic — more data in same format won't fix it |
| Matra confusion | Unicode NFC normalization | Free fix — no retraining needed, reduces tokenizer noise |
| Repetition loops | `repetition_penalty=1.3`, `no_repeat_ngram_size=3` | Decoder artifact from OOD input — training data can't prevent it |

---

## g) Fix Implementation — Repetition Penalty

**Targeted subset:** 6 high-WER hallucination samples (WER > 0.80)

| | WER |
|-|-----|
| Before | 82.35% |
| After | **80.39%** |
| Improvement | **1.96 pp** |

**Key example:**

> **Before:** रूमानियत मेंशास्वास्वास्वास्वास्वास... *(infinite loop)*
>
> **After:** रुमानियत में सांस्क्रतिक नीयती वाद का बड़ा तत्तू था जो गोए थे फिच्ठे और स्लेगल जैसे लेखो को लिया गया

Loop eliminated. Output is imperfect but coherent and usable for downstream NLP.